<a href="https://colab.research.google.com/github/Advanced-Forecasting-Project/NYC-Car-Crashes-Forecasting/blob/main/Forecasting_Project_Polly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYC Motor Vehicle Collisions: Data Cleaning

I will use this notebook to check the crash records and prepare a daily crash-count series for forecasting. I will remove rows only if the crash date is missing or invalid. Will check collision IDs for duplicates and report missing values instead.

## 1. Load and inspect the data

The file is uploaded to this Colab session.

In [20]:
import pandas as pd
url = "https://github.com/Advanced-Forecasting-Project/NYC-Car-Crashes-Forecasting/releases/download/data-v1/Motor_Vehicle_Collisions_-_Crashes.zip"
crashes = pd.read_csv(url, compression="zip", low_memory=False)
print("Rows and columns:", crashes.shape)
print("Columns:", crashes.columns.tolist())
crashes.head()

Rows and columns: (2018963, 29)
Columns: ['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'LOCATION', 'ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3', 'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5', 'COLLISION_ID', 'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2', 'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5']


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/11/2021,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,06/29/2022,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
3,09/11/2021,9:35,BROOKLYN,11208,40.667202,-73.866500,"(40.667202, -73.8665)",NaN,NaN,1211 LORING AVENUE,...,NaN,NaN,NaN,NaN,4456314,Sedan,NaN,NaN,NaN,NaN
4,12/14/2021,8:13,BROOKLYN,11233,40.683304,-73.917274,"(40.683304, -73.917274)",SARATOGA AVENUE,DECATUR STREET,NaN,...,NaN,NaN,NaN,NaN,4486609,NaN,NaN,NaN,NaN,NaN


In [ ]:
crash_dates = pd.to_datetime(crashes['CRASH DATE'], format='%m/%d/%Y', errors='coerce')

print('Rows with an invalid date:', crash_dates.isna().sum())
print('First date:', crash_dates.min())
print('Last date:', crash_dates.max())
print('Rows with a missing collision ID:', crashes['COLLISION_ID'].isna().sum())
print('Repeated collision IDs:', crashes['COLLISION_ID'].duplicated().sum())
print('Exact repeated rows:', crashes.duplicated().sum())

Rows with an invalid date: 0
First date: 2012-07-01 00:00:00
Last date: 2023-08-18 00:00:00
Rows with a missing collision ID: 0
Repeated collision IDs: 0
Exact repeated rows: 0


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": crashes.isna().sum(),
    "missing_percent": (crashes.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
VEHICLE TYPE CODE 5,2010550,99.58
CONTRIBUTING FACTOR VEHICLE 5,2010288,99.57
VEHICLE TYPE CODE 4,1987893,98.46
CONTRIBUTING FACTOR VEHICLE 4,1986822,98.41
VEHICLE TYPE CODE 3,1880762,93.15
CONTRIBUTING FACTOR VEHICLE 3,1875772,92.91
OFF STREET NAME,1686327,83.52
CROSS STREET NAME,755889,37.44
ZIP CODE,628305,31.12
BOROUGH,628067,31.11


In [ ]:
# Keep rows with a valid crash date.
cleaned = crashes.loc[crash_dates.notna()].copy()
cleaned["CRASH DATE"] = crash_dates[crash_dates.notna()]

# Count crashes by day and include dates with no recorded crashes.
daily_crashes = cleaned.groupby("CRASH DATE").size().asfreq("D", fill_value=0)
daily_crashes.name = "crash_count"

# Weekly totals make a second series to compare later.
weekly_crashes = daily_crashes.resample("W").sum()

print("Rows kept:", len(cleaned))
print("Rows removed for invalid dates:", len(crashes) - len(cleaned))
print("Daily series:", daily_crashes.index.min(), "to", daily_crashes.index.max())
print("Days with no recorded crashes:", (daily_crashes == 0).sum())
print("Number of weeks:", len(weekly_crashes))
print("The first and last weeks may be partial weeks.")

# Save the daily and weekly counts for the forecasting work.
daily_crashes.to_csv("nyc_daily_crash_counts.csv")
weekly_crashes.to_csv("nyc_weekly_crash_counts.csv")

Rows kept: 2018963
Rows removed for invalid dates: 0
Daily series: 2012-07-01 00:00:00 to 2023-08-18 00:00:00
Days with no recorded crashes: 0
Number of weeks: 582
The first and last weeks may be partial weeks.


In [ ]:
# Check the first and last monthly totals because these months are incomplete.
monthly_crashes = daily_crashes.resample("MS").sum()
print("First month and crash count:")
print(monthly_crashes.head(1))
print("Last month and crash count:")
print(monthly_crashes.tail(1))

First month and crash count:
CRASH DATE
2012-07-01    16992
Freq: MS, Name: crash_count, dtype: int64
Last month and crash count:
CRASH DATE
2023-08-01    4274
Freq: MS, Name: crash_count, dtype: int64


In [ ]:
# Show the data type saved for each column.
data_types = crashes.dtypes.astype(str).rename("data_type").reset_index()
data_types.columns = ["column", "data_type"]
data_types

,column,data_type
0,CRASH DATE,object
1,CRASH TIME,object
2,BOROUGH,object
3,ZIP CODE,object
4,LATITUDE,float64
5,LONGITUDE,float64
6,LOCATION,object
7,ON STREET NAME,object
8,CROSS STREET NAME,object
9,OFF STREET NAME,object


### What the columns describe

Each row is one police-reported motor vehicle collision. **CRASH DATE** and **CRASH TIME** record when it happened. **BOROUGH**, **ZIP CODE**, coordinates and street names describe where it happened. The injury and fatality columns contain counts. The contributing factor and vehicle type columns describe reported crash details. **COLLISION_ID** identifies a crash record.
For a city wide daily crash forecast, the crash date is used to group records and the number of records becomes the target.

Source: [NYC Open Data, Motor Vehicle Collisions – Crashes](https://www.kaggle.com/datasets/ishmaelkiptoo/motor-vehicle-collisions$0)

In [ ]:
import os

for name in [file_path, "nyc_daily_crash_counts.csv", "nyc_weekly_crash_counts.csv"]:
    size_mb = os.path.getsize(name) / (1024 ** 2)
    print(name, round(size_mb, 2), "MB")

daily_saved = pd.read_csv("nyc_daily_crash_counts.csv")
weekly_saved = pd.read_csv("nyc_weekly_crash_counts.csv")
print("Daily CSV rows and total:", len(daily_saved), daily_saved["crash_count"].sum())
print("Weekly CSV rows and total:", len(weekly_saved), weekly_saved["crash_count"].sum())
print("Raw crash rows:", len(crashes))

/content/Motor_Vehicle_Collisions_-_Crashes.csv 409.68 MB
nyc_daily_crash_counts.csv 0.06 MB
nyc_weekly_crash_counts.csv 0.01 MB
Daily CSV rows and total: 4066 2018963
Weekly CSV rows and total: 582 2018963
Raw crash rows: 2018963
